# Datenbereinigung und Preprocessing

In diesem Notebook wird der Datensatz auf Basis der Ergebnisse der explorativen Datenanalyse für die spätere Modellierung vorbereitet.

Dabei werden Spaltennamen vereinheitlicht, vollständige Duplikate behandelt und redundante Merkmale entfernt. Modellabhängige Transformationen wie Skalierung oder One-Hot-Encoding werden erst nach dem Train/Test Split durchgeführt, um Data Leakage zu vermeiden.

In [17]:
import pandas as pd
import numpy as np

In [18]:
df = pd.read_csv("../data/raw/iranian_churn.csv")

df.shape

(3150, 14)

## Spaltennamen vereinheitlichen

In [19]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"\s+", "_", regex=True)
)

df.columns.tolist()

['call_failure',
 'complains',
 'subscription_length',
 'charge_amount',
 'seconds_of_use',
 'frequency_of_use',
 'frequency_of_sms',
 'distinct_called_numbers',
 'age_group',
 'tariff_plan',
 'status',
 'age',
 'customer_value',
 'churn']

## Vollständige Duplikate behandeln

In der explorativen Analyse wurden 300 vollständig identische Zeilen gefunden. Da keine eindeutige Kunden-ID vorhanden ist, lässt sich nicht sicher feststellen, ob diese Beobachtungen tatsächlich dieselben Kunden repräsentieren.

Für die Modellierung werden vollständig identische Datensätze dennoch nur einmal berücksichtigt. Dadurch wird verhindert, dass identische Beobachtungen gleichzeitig im Trainings- und Testdatensatz auftreten und die Modellbewertung künstlich verbessern.

In [20]:
rows_before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

rows_after = len(df)

print(f"Zeilen vorher: {rows_before}")
print(f"Zeilen nachher: {rows_after}")
print(f"Entfernte Duplikate: {rows_before - rows_after}")

Zeilen vorher: 3150
Zeilen nachher: 2850
Entfernte Duplikate: 300


Featureprofile, die bei identischen Eingangsmerkmalen unterschiedliche Churn-Werte besitzen, werden nicht entfernt. Ohne eindeutige Kunden-ID gibt es keine ausreichende Grundlage, diese Beobachtungen als Datenfehler einzustufen.

## Redundantes Merkmal entfernen

`age` und `age_group` weisen im Datensatz eine eindeutige 1:1-Zuordnung auf und enthalten damit dieselbe Information. Um redundante Features zu vermeiden, wird `age_group` entfernt und `age` beibehalten.

In [21]:
df = df.drop(columns=["age_group"])

In [22]:
df.shape

(2850, 13)

In [23]:
print("Fehlende Werte:")
print(df.isnull().sum().sum())

print("\nVollständige Duplikate:")
print(df.duplicated().sum())

print("\nDimension:")
print(df.shape)

Fehlende Werte:
0

Vollständige Duplikate:
0

Dimension:
(2850, 13)


## Zielvariable nach der Datenbereinigung

In [24]:
churn_distribution = df["churn"].value_counts()
churn_percentage = df["churn"].value_counts(normalize=True).mul(100).round(2)

print("Absolute Häufigkeiten:")
print(churn_distribution)

print("\nProzentuale Verteilung:")
print(churn_percentage)

Absolute Häufigkeiten:
churn
0    2404
1     446
Name: count, dtype: int64

Prozentuale Verteilung:
churn
0    84.35
1    15.65
Name: proportion, dtype: float64


## Feature-Typen definieren

Für die spätere Modellierung werden die Eingangsvariablen nach ihrer fachlichen Bedeutung in numerische und kategoriale Merkmale aufgeteilt. Die Variable `age` wird trotz ihres numerischen Datentyps kategorial behandelt, da sie im Datensatz lediglich fünf diskrete Altersgruppen repräsentiert.

In [25]:
numeric_features = [
    "call_failure",
    "subscription_length",
    "charge_amount",
    "seconds_of_use",
    "frequency_of_use",
    "frequency_of_sms",
    "distinct_called_numbers",
    "customer_value"
]

categorical_features = [
    "complains",
    "tariff_plan",
    "status",
    "age"
]

target = "churn"

In [26]:
print(f"Numerische Features: {len(numeric_features)}")
print(f"Kategoriale Features: {len(categorical_features)}")
print(f"Gesamtzahl Features: {len(numeric_features) + len(categorical_features)}")

Numerische Features: 8
Kategoriale Features: 4
Gesamtzahl Features: 12


In [27]:
X = df.drop(columns=[target])
y = df[target]

In [28]:
print("X:", X.shape)
print("y:", y.shape)

X: (2850, 12)
y: (2850,)


## Bereinigten Datensatz speichern

Der bereinigte Datensatz wird getrennt von den unveränderten Rohdaten gespeichert. Dadurch bleibt der ursprüngliche Datensatz jederzeit reproduzierbar erhalten.

In [29]:
output_path = "../data/processed/iranian_churn_cleaned.csv"

df.to_csv(output_path, index=False)

print(f"Bereinigter Datensatz gespeichert: {output_path}")
print(f"Dimension: {df.shape}")

Bereinigter Datensatz gespeichert: ../data/processed/iranian_churn_cleaned.csv
Dimension: (2850, 13)


In [30]:
print("Absolute Häufigkeiten:")
print(churn_distribution)

print("\nProzentuale Verteilung:")
print(churn_percentage)

Absolute Häufigkeiten:
churn
0    2404
1     446
Name: count, dtype: int64

Prozentuale Verteilung:
churn
0    84.35
1    15.65
Name: proportion, dtype: float64


In [31]:
print("X:", X.shape)
print("y:", y.shape)

X: (2850, 12)
y: (2850,)


## Validierung des bereinigten Datensatzes

Der gespeicherte Datensatz wird erneut eingelesen, um zu überprüfen, ob die Bereinigung korrekt und reproduzierbar gespeichert wurde.

In [32]:
df_check = pd.read_csv("../data/processed/iranian_churn_cleaned.csv")

print("Dimension:", df_check.shape)
print("Fehlende Werte:", df_check.isnull().sum().sum())
print("Duplikate:", df_check.duplicated().sum())
print("Spalten:")
print(df_check.columns.tolist())

Dimension: (2850, 13)
Fehlende Werte: 0
Duplikate: 0
Spalten:
['call_failure', 'complains', 'subscription_length', 'charge_amount', 'seconds_of_use', 'frequency_of_use', 'frequency_of_sms', 'distinct_called_numbers', 'tariff_plan', 'status', 'age', 'customer_value', 'churn']
